# SUB004 -- Train-Data Template-Based RNA 3D Prediction

**Competition**: Stanford RNA 3D Folding Part 2  
**Metric**: TM-score (higher is better, best-of-5)  
**Lineage**: IT001 → IT002 → SUB001-SUB003 (all failed) → SUB004

Key change: Use competition training data (5716 sequences) as template library.  
No internet, no external datasets, no neural networks needed.

In [ ]:
import os
import sys
import time
import zlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 60)
print("SUB004: Train-Data Template-Based RNA 3D Prediction")
print("=" * 60)

# Diagnostic: list input directory
INPUT_BASE = "/kaggle/input"
if os.path.isdir(INPUT_BASE):
    print(f"\nContents of {INPUT_BASE}:")
    for item in sorted(os.listdir(INPUT_BASE)):
        full = os.path.join(INPUT_BASE, item)
        if os.path.isdir(full):
            sub_items = os.listdir(full)
            print(f"  {item}/ ({len(sub_items)} items)")
            for si in sorted(sub_items)[:10]:
                print(f"    {si}")
            if len(sub_items) > 10:
                print(f"    ... and {len(sub_items) - 10} more")
        else:
            print(f"  {item} ({os.path.getsize(full)} bytes)")
else:
    print(f"WARNING: {INPUT_BASE} does not exist")
    print(f"CWD: {os.getcwd()}")
    print(f"CWD contents: {os.listdir('.')}")

In [ ]:
# ============================================================
# Data Loading - robust path detection
# ============================================================
CANDIDATE_BASES = [
    "/kaggle/input/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-part-2",
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
]

DATA_PATH = None
for p in CANDIDATE_BASES:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "test_sequences.csv")):
        DATA_PATH = p
        break
    elif os.path.isdir(p):
        print(f"Directory {p} exists but test_sequences.csv not found")
        print(f"  Contents: {os.listdir(p)[:10]}")

if DATA_PATH is None:
    # Last resort: search all input directories for test_sequences.csv
    if os.path.isdir(INPUT_BASE):
        for d in os.listdir(INPUT_BASE):
            fp = os.path.join(INPUT_BASE, d, "test_sequences.csv")
            if os.path.isfile(fp):
                DATA_PATH = os.path.join(INPUT_BASE, d)
                print(f"Found data at: {DATA_PATH}")
                break

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find competition data. "
        f"Searched: {CANDIDATE_BASES}. "
        f"Available dirs: {os.listdir(INPUT_BASE) if os.path.isdir(INPUT_BASE) else 'N/A'}"
    )

print(f"\nUsing data path: {DATA_PATH}")
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

train_seqs = pd.read_csv(os.path.join(DATA_PATH, "train_sequences.csv"))
test_seqs = pd.read_csv(os.path.join(DATA_PATH, "test_sequences.csv"))
train_labels = pd.read_csv(os.path.join(DATA_PATH, "train_labels.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_PATH, "sample_submission.csv"))

print(f"Train sequences: {len(train_seqs)}")
print(f"Test sequences:  {len(test_seqs)}")
print(f"Train labels:    {len(train_labels)}")
print(f"Sample sub:      {len(sample_sub)}")

In [ ]:
# ============================================================
# Build template library from training data
# ============================================================
def process_labels(labels_df):
    """Extract C1' coordinates per target from train_labels."""
    coords_dict = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes, sort=False):
        sorted_group = group.sort_values("resid")
        coords_dict[id_prefix] = sorted_group[["x_1", "y_1", "z_1"]].values.astype(np.float64)
    return coords_dict

print("Processing training labels...")
t0 = time.time()
train_coords_dict = process_labels(train_labels)
print(f"Processed {len(train_coords_dict)} targets in {time.time()-t0:.1f}s")

# Build template arrays
TRAIN_IDS, TRAIN_SEQS, TRAIN_COORDS, TRAIN_LENS = [], [], [], []

for r in train_seqs.itertuples(index=False):
    tid = r.target_id
    if tid not in train_coords_dict:
        continue
    seq = r.sequence
    coords = train_coords_dict[tid]
    if len(coords) != len(seq):
        continue
    # Skip templates with all-zero or invalid coords
    if np.all(coords == 0) or np.any(np.isnan(coords)):
        continue
    TRAIN_IDS.append(tid)
    TRAIN_SEQS.append(seq)
    TRAIN_COORDS.append(coords)
    TRAIN_LENS.append(len(seq))

TRAIN_LENS = np.array(TRAIN_LENS, dtype=np.int32)
print(f"Template library: {len(TRAIN_IDS)} templates with valid coordinates")
print(f"  Length range: {TRAIN_LENS.min()}-{TRAIN_LENS.max()}, mean={TRAIN_LENS.mean():.0f}")

In [ ]:
# ============================================================
# K-mer indexing for fast template retrieval
# ============================================================
KMER_K = 5
PREFILTER_TOP = 200
ALIGN_TOP = 20

_BASE2 = {"A": 0, "C": 1, "G": 2, "U": 3}

def kmer_set_2bit(seq, k=KMER_K):
    if len(seq) < k:
        return frozenset()
    mask = (1 << (2 * k)) - 1
    code = 0
    out = set()
    valid = True
    for i in range(k):
        b = _BASE2.get(seq[i])
        if b is None:
            valid = False
            break
        code = ((code << 2) | b) & mask
    if valid:
        out.add(code)
        for i in range(k, len(seq)):
            b = _BASE2.get(seq[i])
            if b is None:
                continue
            code = ((code << 2) | b) & mask
            out.add(code)
    return frozenset(out)

print("Building k-mer index...")
t0 = time.time()
TRAIN_KMERS = [kmer_set_2bit(s, KMER_K) for s in TRAIN_SEQS]
print(f"K-mer index built in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# Alignment and Coordinate Transfer
# ============================================================
def needleman_wunsch(seq_a, seq_b, match=2, mismatch=-1, gap=-2):
    """Optimized NW with score-only fast path for long sequences."""
    n, m = len(seq_a), len(seq_b)
    
    # For very long sequences, use banded approach
    if n * m > 25_000_000:  # > 5000x5000
        return _banded_nw(seq_a, seq_b, match, mismatch, gap, band_width=100)
    
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    dp[0, :] = np.arange(m + 1) * gap
    dp[:, 0] = np.arange(n + 1) * gap
    
    a_arr = np.array(list(seq_a))
    b_arr = np.array(list(seq_b))
    
    for i in range(1, n + 1):
        scores = np.where(b_arr == a_arr[i - 1], match, mismatch)
        diag = dp[i - 1, :-1] + scores
        up = dp[i - 1, 1:] + gap
        dp[i, 1:] = np.maximum(diag, up)
        # Left dependency scan
        for j in range(1, m + 1):
            dp[i, j] = max(dp[i, j], dp[i, j - 1] + gap)
    
    # Traceback
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match if seq_a[i - 1] == seq_b[j - 1] else mismatch
            if dp[i, j] == dp[i - 1, j - 1] + s:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i - 1, j] + gap:
            i -= 1
        else:
            j -= 1
    
    return map_a_to_b, dp[n, m]


def _banded_nw(seq_a, seq_b, match=2, mismatch=-1, gap=-2, band_width=100):
    """Banded NW for long sequences - only compute cells near diagonal."""
    n, m = len(seq_a), len(seq_b)
    
    NEGINF = -999999
    dp = {}
    dp[(0, 0)] = 0
    for j in range(1, min(band_width + 1, m + 1)):
        dp[(0, j)] = j * gap
    for i in range(1, min(band_width + 1, n + 1)):
        dp[(i, 0)] = i * gap
    
    ratio = m / n if n > 0 else 1.0
    
    for i in range(1, n + 1):
        center_j = int(i * ratio)
        j_lo = max(1, center_j - band_width)
        j_hi = min(m, center_j + band_width)
        for j in range(j_lo, j_hi + 1):
            s = match if seq_a[i - 1] == seq_b[j - 1] else mismatch
            d = dp.get((i - 1, j - 1), NEGINF) + s
            u = dp.get((i - 1, j), NEGINF) + gap
            l = dp.get((i, j - 1), NEGINF) + gap
            dp[(i, j)] = max(d, u, l)
    
    score = dp.get((n, m), NEGINF)
    
    # Traceback
    map_a_to_b = {}
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match if seq_a[i - 1] == seq_b[j - 1] else mismatch
            if dp.get((i, j)) == dp.get((i - 1, j - 1), NEGINF) + s:
                map_a_to_b[i - 1] = j - 1
                i -= 1
                j -= 1
                continue
        if i > 0 and dp.get((i, j)) == dp.get((i - 1, j), NEGINF) + gap:
            i -= 1
        elif j > 0:
            j -= 1
        else:
            break
    
    return map_a_to_b, score


def nw_score_only(seq_a, seq_b, match=2, mismatch=-1.5, gap=-2):
    """Fast score-only NW using 1D array (no traceback)."""
    n, m = len(seq_a), len(seq_b)
    if n * m > 50_000_000:
        # Too long, use kmer Jaccard as proxy
        return _kmer_score_proxy(seq_a, seq_b)
    
    prev = np.arange(m + 1, dtype=np.float32) * gap
    b_arr = np.array(list(seq_b))
    
    for i in range(1, n + 1):
        curr = np.empty(m + 1, dtype=np.float32)
        curr[0] = i * gap
        scores = np.where(b_arr == seq_a[i - 1], match, mismatch)
        diag = prev[:-1] + scores
        up = prev[1:] + gap
        curr[1:] = np.maximum(diag, up)
        for j in range(1, m + 1):
            curr[j] = max(curr[j], curr[j - 1] + gap)
        prev = curr
    
    return float(prev[m])


def _kmer_score_proxy(seq_a, seq_b, k=5):
    """Fast kmer-based similarity score for very long sequences."""
    ka = kmer_set_2bit(seq_a, k)
    kb = kmer_set_2bit(seq_b, k)
    if not ka or not kb:
        return 0.0
    inter = len(ka & kb)
    union = len(ka | kb)
    jaccard = inter / union if union > 0 else 0.0
    return jaccard * 2.0 * min(len(seq_a), len(seq_b))


def transfer_coordinates(query_seq, template_seq, template_coords, alignment_map):
    """Transfer template coordinates to query using alignment mapping."""
    qlen = len(query_seq)
    result = np.full((qlen, 3), np.nan, dtype=np.float64)
    
    for q_pos, t_pos in alignment_map.items():
        if q_pos < qlen and t_pos < len(template_coords):
            result[q_pos] = template_coords[t_pos]
    
    # Interpolate gaps
    mapped = sorted(alignment_map.keys())
    if not mapped:
        return _linear_chain(qlen)
    
    for i in range(qlen):
        if not np.isnan(result[i, 0]):
            continue
        prev_v = next((j for j in range(i - 1, -1, -1) if not np.isnan(result[j, 0])), -1)
        next_v = next((j for j in range(i + 1, qlen) if not np.isnan(result[j, 0])), -1)
        if prev_v >= 0 and next_v >= 0:
            w = (i - prev_v) / (next_v - prev_v)
            result[i] = (1 - w) * result[prev_v] + w * result[next_v]
        elif prev_v >= 0:
            result[i] = result[prev_v] + [5.95, 0, 0]
        elif next_v >= 0:
            result[i] = result[next_v] - [5.95, 0, 0]
        else:
            result[i] = [i * 5.95, 0, 0]
    
    return np.nan_to_num(result)


def _linear_chain(length, spacing=5.95):
    """Generate a straight-line chain as fallback."""
    coords = np.zeros((length, 3), dtype=np.float64)
    for i in range(1, length):
        coords[i] = coords[i - 1] + [spacing, 0, 0]
    return coords


print("Alignment and transfer functions loaded.")

In [ ]:
# ============================================================
# Template Retrieval
# ============================================================
def find_similar_sequences(query_seq, top_n=ALIGN_TOP):
    """Find most similar templates using length filter + kmer prefilter + optional NW."""
    qlen = len(query_seq)
    qkm = kmer_set_2bit(query_seq, KMER_K)
    qkm_len = len(qkm)
    
    if len(TRAIN_IDS) == 0:
        return []
    
    maxL = np.maximum(TRAIN_LENS, qlen)
    keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.30
    idxs = np.where(keep)[0]
    if idxs.size == 0:
        keep = (np.abs(TRAIN_LENS - qlen) / maxL) <= 0.60
        idxs = np.where(keep)[0]
    if idxs.size == 0:
        return []
    
    scored = []
    for i in idxs:
        tkm = TRAIN_KMERS[i]
        if qkm_len == 0 or len(tkm) == 0:
            jac = 0.0
        else:
            inter = len(qkm & tkm)
            union = qkm_len + len(tkm) - inter
            jac = inter / union if union else 0.0
        scored.append((jac, int(i)))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    top_candidates = scored[:PREFILTER_TOP]
    
    NW_CELL_LIMIT = 5_000_000
    sims = []
    for jac, i in top_candidates:
        t_seq = TRAIN_SEQS[i]
        if qlen * len(t_seq) < NW_CELL_LIMIT:
            raw = nw_score_only(query_seq, t_seq)
            denom = 2.0 * min(qlen, len(t_seq))
            norm = float(raw / denom) if denom > 0 else jac
        else:
            norm = jac
        sims.append((TRAIN_IDS[i], t_seq, norm, TRAIN_COORDS[i], i))
    
    sims.sort(key=lambda x: x[2], reverse=True)
    return sims[:top_n]

print("Template retrieval ready.")

In [ ]:
# ============================================================
# Stoichiometry and Chain Handling
# ============================================================
def parse_fasta(fasta_content):
    out = {}
    cur = None
    seq_parts = []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if cur is not None:
                out[cur] = "".join(seq_parts)
            cur = line[1:].split()[0]
            seq_parts = []
        else:
            seq_parts.append(line.replace(" ", ""))
    if cur is not None:
        out[cur] = "".join(seq_parts)
    return out


def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(";"):
        parts = part.split(":")
        if len(parts) == 2:
            out.append((parts[0].strip(), int(parts[1])))
    return out


def get_chain_segments(row):
    """Get (start, end) segments for each chain copy."""
    seq = row["sequence"]
    stoich = row.get("stoichiometry", "")
    all_seq = row.get("all_sequences", "")
    
    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip() == "" or str(all_seq).strip() == "":
        return [(0, len(seq))]
    
    try:
        chain_dict = parse_fasta(all_seq)
        order = parse_stoichiometry(stoich)
        segs = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                L = len(base)
                segs.append((pos, pos + L))
                pos += L
        if pos != len(seq):
            return [(0, len(seq))]
        return segs
    except Exception:
        return [(0, len(seq))]


# Build segment maps
test_segs_map = {}
for _, r in test_seqs.iterrows():
    test_segs_map[r["target_id"]] = get_chain_segments(r)

print(f"Chain segments computed for {len(test_segs_map)} test targets")
for tid, segs in list(test_segs_map.items())[:3]:
    print(f"  {tid}: {len(segs)} chain(s), total {sum(e-s for s,e in segs)} residues")

In [ ]:
# ============================================================
# RNA Structural Constraints
# ============================================================
def adaptive_rna_constraints(coordinates, segments, confidence=1.0, passes=2):
    """Apply physical constraints: bond distances, steric repulsion, smoothing."""
    coords = coordinates.copy()
    strength = 0.75 * (1.0 - min(confidence, 0.90))
    strength = max(strength, 0.02)
    
    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                continue
            
            # C1'-C1' bond distance constraint (~5.95 Angstrom)
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.22 * strength)
            X[:-1] -= adj
            X[1:] += adj
            
            # Second-neighbor distance constraint
            if L >= 5:
                d2 = X[2:] - X[:-2]
                dist2 = np.linalg.norm(d2, axis=1) + 1e-6
                target2 = 10.2
                scale2 = (target2 - dist2) / dist2
                adj2 = (d2 * scale2[:, None]) * (0.10 * strength)
                X[:-2] -= adj2
                X[2:] += adj2
            
            # Laplacian smoothing
            if L >= 5:
                lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
                X[1:-1] += (0.06 * strength) * lap
            
            # Steric repulsion for longer chains
            if L >= 25:
                k = min(L, 160)
                idx = np.linspace(0, L - 1, k).astype(int) if k < L else np.arange(L)
                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])
                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.015 * strength) * vec
            
            coords[s:e] = X
    
    return coords

print("Structural constraints loaded.")

In [ ]:
# ============================================================
# Diversity Transforms for Best-of-5
# ============================================================
def stable_hash32(s):
    return zlib.adler32(s.encode("utf-8")) & 0xFFFFFFFF


def _rotmat(axis, ang):
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    x, y, z = axis
    c, s = np.cos(ang), np.sin(ang)
    C = 1.0 - c
    return np.array([
        [c + x*x*C, x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s, c + y*y*C, y*z*C - x*s],
        [z*x*C - y*s, z*y*C + x*s, c + z*z*C]
    ], dtype=float)


def apply_hinge(coords, seg, rng, max_angle_deg=25):
    s, e = seg
    L = e - s
    if L < 30:
        return coords
    pivot = s + int(rng.integers(10, L - 10))
    axis = rng.normal(size=3)
    ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
    R = _rotmat(axis, ang)
    X = coords.copy()
    p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X


def jitter_chains(coords, segments, rng, max_angle_deg=12, max_trans=1.5):
    X = coords.copy()
    global_center = X.mean(axis=0, keepdims=True)
    for (s, e) in segments:
        axis = rng.normal(size=3)
        ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
        R = _rotmat(axis, ang)
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0.0, max_trans))
        c = X[s:e].mean(axis=0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    X -= X.mean(axis=0, keepdims=True) - global_center
    return X


def smooth_wiggle(coords, segments, rng, amp=0.8):
    X = coords.copy()
    for (s, e) in segments:
        L = e - s
        if L < 20:
            continue
        n_ctrl = 6
        ctrl_x = np.linspace(0, L - 1, n_ctrl)
        ctrl_disp = rng.normal(0, amp, size=(n_ctrl, 3))
        t = np.arange(L)
        disp = np.vstack([np.interp(t, ctrl_x, ctrl_disp[:, k]) for k in range(3)]).T
        X[s:e] += disp
    return X


print("Diversity transforms loaded.")

In [ ]:
# ============================================================
# Prediction Pipeline
# ============================================================
def predict_rna_structures(tid, seq, n_predictions=5):
    """Predict n_predictions diverse 3D structures for a given RNA sequence."""
    segments = test_segs_map.get(tid, [(0, len(seq))])
    cands = find_similar_sequences(query_seq=seq, top_n=ALIGN_TOP)
    predictions = []
    used = set()
    
    for i in range(n_predictions):
        seed = (stable_hash32(tid) + i * 10007) & 0xFFFFFFFF
        rng = np.random.default_rng(seed)
        
        if not cands:
            coords = _linear_chain(len(seq))
            predictions.append(coords)
            continue
        
        # Select template
        if i == 0:
            t_id, t_seq, sim, t_coords, _ = cands[0]
        else:
            K = min(12, len(cands))
            sims = np.array([cands[k][2] for k in range(K)], float)
            w = np.exp((sims - sims.max()) / 0.08)
            for k in range(K):
                if cands[k][0] in used:
                    w[k] *= 0.10
            w = w / (w.sum() + 1e-12)
            k = int(rng.choice(np.arange(K), p=w))
            t_id, t_seq, sim, t_coords, _ = cands[k]
        
        used.add(t_id)
        
        # Coordinate transfer
        alignment_map, _ = needleman_wunsch(seq, t_seq)
        adapted = transfer_coordinates(seq, t_seq, t_coords, alignment_map)
        
        # Apply diversity transform
        if i == 0:
            X = adapted
        elif i == 1:
            noise_std = max(0.01, (0.40 - sim) * 0.06)
            X = adapted + rng.normal(0, noise_std, adapted.shape)
        elif i == 2:
            longest = max(segments, key=lambda se: se[1] - se[0])
            X = apply_hinge(adapted, longest, rng, max_angle_deg=22)
        elif i == 3:
            X = jitter_chains(adapted, segments, rng, max_angle_deg=10, max_trans=1.0)
        else:
            X = smooth_wiggle(adapted, segments, rng, amp=0.7)
        
        # Apply constraints
        refined = adaptive_rna_constraints(X, segments, confidence=sim, passes=2)
        predictions.append(refined)
    
    return predictions

print("Prediction pipeline ready.")

In [ ]:
# ============================================================
# Generate Predictions for All Test Targets
# ============================================================
print(f"\nGenerating predictions for {len(test_seqs)} test targets...")
start_time = time.time()

dfs = []

for idx, r in enumerate(test_seqs.itertuples(index=False)):
    tid = r.target_id
    seq = r.sequence
    L = len(seq)
    
    t0 = time.time()
    preds = predict_rna_structures(tid, seq, n_predictions=5)
    elapsed = time.time() - t0
    
    if idx < 5 or idx % 10 == 0:
        print(f"  [{idx+1}/{len(test_seqs)}] {tid} (L={L}) -> {elapsed:.1f}s")
    
    data = {
        "ID": [f"{tid}_{j}" for j in range(1, L + 1)],
        "resname": list(seq),
        "resid": np.arange(1, L + 1, dtype=np.int32),
    }
    for i in range(5):
        data[f"x_{i+1}"] = preds[i][:, 0].astype(np.float32)
        data[f"y_{i+1}"] = preds[i][:, 1].astype(np.float32)
        data[f"z_{i+1}"] = preds[i][:, 2].astype(np.float32)
    
    dfs.append(pd.DataFrame(data))

total_time = time.time() - start_time
print(f"\nAll predictions generated in {total_time:.1f}s")

In [ ]:
# ============================================================
# Build and Save Submission
# ============================================================
sub = pd.concat(dfs, ignore_index=True)

cols = ["ID", "resname", "resid"] + [f"{c}_{i}" for i in range(1, 6) for c in ["x", "y", "z"]]
coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]

sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
sub = sub.fillna(0.0)

output_path = os.path.join(WORK, "submission.csv")
sub[cols].to_csv(output_path, index=False)

print(f"Submission saved to {output_path}")
print(f"Shape: {sub.shape}")
print(f"Expected: {len(sample_sub)} rows")
print(f"NaN values: {sub[coord_cols].isna().sum().sum()}")
print(f"Zero values: {(sub[coord_cols] == 0).sum().sum()}")
print()
print(sub.head())
print()
print("=" * 60)
print("SUB004 Pipeline Complete")
print("=" * 60)
print(f"  Templates used  : {len(TRAIN_IDS)}")
print(f"  Test targets    : {len(test_seqs)}")
print(f"  Submission rows : {len(sub)}")
print(f"  Total time      : {total_time:.1f}s")
print("=" * 60)